In [1]:
import pandas as pd

df = pd.read_csv('../data/obesity_level.csv')

print("Filas y columnas:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

df.head()

Filas y columnas: (20758, 18)

Columnas:
['id', 'Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', '0be1dad']


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,0be1dad
0,0,Male,24.443011,1.699998,81.669950,1,1,2.000000,2.983297,Sometimes,0,2.763573,0,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,1,1,2.000000,3.000000,Frequently,0,2.000000,0,1.000000,1.000000,0,Automobile,0rmal_Weight
2,2,Female,18.000000,1.711460,50.165754,1,1,1.880534,1.411685,Sometimes,0,1.910378,0,0.866045,1.673584,0,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,1,1,3.000000,3.000000,Sometimes,0,1.674061,0,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,1,1,2.679664,1.971472,Sometimes,0,1.979848,0,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [2]:
print("Column names and data types:")
print(df.dtypes)

print("\nNull values per column:")
print(df.isnull().sum())

print("\nGender distribution:")
print(df['Gender'].value_counts())

Column names and data types:
id                                  int64
Gender                                str
Age                               float64
Height                            float64
Weight                            float64
family_history_with_overweight      int64
FAVC                                int64
FCVC                              float64
NCP                               float64
CAEC                                  str
SMOKE                               int64
CH2O                              float64
SCC                                 int64
FAF                               float64
TUE                               float64
CALC                                  str
MTRANS                                str
0be1dad                               str
dtype: object

Null values per column:
id                                0
Gender                            0
Age                               0
Height                            0
Weight                          

In [4]:
print("Exact column names:")
print(df.columns.tolist())

print("\nTarget variable unique values:")
print(df['0be1dad'].value_counts())

Exact column names:
['id', 'Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', '0be1dad']

Target variable unique values:
0be1dad
Obesity_Type_III       4046
Obesity_Type_II        3248
0rmal_Weight           3082
Obesity_Type_I         2910
Insufficient_Weight    2523
Overweight_Level_II    2522
Overweight_Level_I     2427
Name: count, dtype: int64


In [5]:
for col in ['Gender', 'CAEC', 'CALC', 'MTRANS']:
    print(f"\n{col}:")
    print(df[col].unique())


Gender:
<StringArray>
['Male', 'Female']
Length: 2, dtype: str

CAEC:
<StringArray>
['Sometimes', 'Frequently', '0', 'Always']
Length: 4, dtype: str

CALC:
<StringArray>
['Sometimes', '0', 'Frequently']
Length: 3, dtype: str

MTRANS:
<StringArray>
['Public_Transportation', 'Automobile', 'Walking', 'Motorbike', 'Bike']
Length: 5, dtype: str


In [6]:
df = df.rename(columns={'0be1dad': 'NObeyesdad'})
df['NObeyesdad'] = df['NObeyesdad'].replace('0rmal_Weight', 'Normal_Weight')
df['CAEC'] = df['CAEC'].replace('0', 'No')
df['CALC'] = df['CALC'].replace('0', 'No')

print(df['NObeyesdad'].value_counts())
print(df['CAEC'].value_counts())
print(df['CALC'].value_counts())

NObeyesdad
Obesity_Type_III       4046
Obesity_Type_II        3248
Normal_Weight          3082
Obesity_Type_I         2910
Insufficient_Weight    2523
Overweight_Level_II    2522
Overweight_Level_I     2427
Name: count, dtype: int64
CAEC
Sometimes     17529
Frequently     2472
Always          478
No              279
Name: count, dtype: int64
CALC
Sometimes     15066
No             5163
Frequently      529
Name: count, dtype: int64


## Data Quality Finding: Corrupted "No"/"Yes" Text Replacement

While exploring the target variable, an unexpected value `0be1dad` appeared 
instead of the expected column name `NObeyesdad`, along with a category 
value `0rmal_Weight` instead of `Normal_Weight`.

Investigation revealed a systematic pattern: the substrings "No" and "Yes" 
had been globally replaced with "0" and "1" somewhere upstream in this 
dataset's distribution — likely an automated cleaning step that didn't 
account for these substrings appearing inside other words (e.g., "**No**rmal" 
and "N**O**be**yes**dad" both contain "No"/"yes" and were affected twice in 
the same word).

This same pattern explains why binary columns like `family_history_with_overweight`, 
`FAVC`, `SMOKE`, and `SCC` appear as 0/1 instead of "No"/"Yes" — a fortunate 
side effect that actually left them in a usable numeric format. However, 
`CAEC`, `CALC`, and the target column required manual correction:
- Renamed `0be1dad` → `NObeyesdad`
- Fixed `0rmal_Weight` → `Normal_Weight`
- Fixed stray `"0"` values in `CAEC` and `CALC` → `"No"`

This is a reminder to never assume column names or category values are 
clean, even from a well-known, widely-used public dataset.

In [7]:
df_women = df[df['Gender'] == 'Female'].copy()

print("Total women:", len(df_women))
print("\nWeight category distribution (women only):")
print(df_women['NObeyesdad'].value_counts())

Total women: 10422

Weight category distribution (women only):
NObeyesdad
Obesity_Type_III       4041
Normal_Weight          1660
Insufficient_Weight    1621
Obesity_Type_I         1267
Overweight_Level_I     1070
Overweight_Level_II     755
Obesity_Type_II           8
Name: count, dtype: int64


In [8]:
print("Obesity_Type_II by gender:")
print(df[df['NObeyesdad'] == 'Obesity_Type_II']['Gender'].value_counts())

print("\nFull distribution by gender (crosstab):")
print(pd.crosstab(df['Gender'], df['NObeyesdad']))

Obesity_Type_II by gender:
Gender
Male      3240
Female       8
Name: count, dtype: int64

Full distribution by gender (crosstab):
NObeyesdad  Insufficient_Weight  Normal_Weight  Obesity_Type_I  \
Gender                                                           
Female                     1621           1660            1267   
Male                        902           1422            1643   

NObeyesdad  Obesity_Type_II  Obesity_Type_III  Overweight_Level_I  \
Gender                                                              
Female                    8              4041                1070   
Male                   3240                 5                1357   

NObeyesdad  Overweight_Level_II  
Gender                           
Female                      755  
Male                       1767  


## Data Quality Finding: Gender Imbalance in Obesity_Type_II

While exploring the target variable by gender, a striking pattern emerged: 
`Obesity_Type_II` is almost exclusively male (3,240 of 3,248 cases, 99.8%), 
while the adjacent category `Obesity_Type_III` is almost exclusively female 
(4,041 of 4,046 cases, 99.9%).

This is likely a consequence of this dataset's generation method — it is 
partly synthetic (built using SMOTE on a smaller real survey sample), and 
the height/weight profiles that originally defined "Obesity_Type_II" in 
the source data appear to have been predominantly male, a pattern that 
SMOTE preserved and amplified rather than correcting.

**Practical impact:** among women specifically, `Obesity_Type_II` has only 
8 cases — too few to treat as a reliable category for statistical testing 
or modeling. For the women-focused analysis in this project, these 8 cases 
are excluded, leaving **6 weight categories**: Insufficient_Weight, 
Normal_Weight, Overweight_Level_I, Overweight_Level_II, Obesity_Type_I, 
and Obesity_Type_III.

This is disclosed here rather than silently dropped, since it reflects a 
real structural bias in the dataset worth being transparent about.